# 🏋️ 헬스 기구 인식 파이프라인 — Phase 1~3 (v4)
### Phase 1: 심층 EDA | Phase 2: YOLO26 학습 (다중 실험) | Phase 3: 성능 검증 & 비교

> **사전 조건**: Phase 0 (데이터 정제) + Workout Pose 이미지 삭제 완료 (`remove_workout_pose.py`)  
> **클래스 수**: 28개 (Phase 0에서 4개 제거 + Phase 1에서 5개 추가 제거)  
> **실험 설계**:
> - 베이스라인: yolo26s / 15 epochs (기존 결과 유지)
> - 실험 A: yolo26s / 100 epochs / patience 20
> - 실험 B: yolo26m / 100 epochs / patience 20
> - 실험 C: yolo26s + small.pt pretrained / 100 epochs

---
## 0. 환경 설정 및 경로 지정

In [1]:
import os
import sys
import yaml
import glob
import random
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from collections import Counter, defaultdict
from matplotlib.patches import Patch

# ──────────────────────────────────────────────
# ⚙️ 경로 설정
# ──────────────────────────────────────────────
DATASET_ROOT = Path(".")

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"
VALID_IMAGES = DATASET_ROOT / "valid" / "images"
VALID_LABELS = DATASET_ROOT / "valid" / "labels"
TEST_IMAGES  = DATASET_ROOT / "test"  / "images"
TEST_LABELS  = DATASET_ROOT / "test"  / "labels"
DATA_YAML    = DATASET_ROOT / "data.yaml"

# data.yaml 로드
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg["names"]
NC = data_cfg["nc"]

print(f"데이터셋 경로: {DATASET_ROOT.resolve()}")
print(f"클래스 수: {NC}")
print(f"클래스 목록: {CLASS_NAMES}")

# 경로 검증
for p, name in [(TRAIN_IMAGES, "train/images"), (TRAIN_LABELS, "train/labels"),
                (VALID_IMAGES, "valid/images"), (VALID_LABELS, "valid/labels"),
                (TEST_IMAGES, "test/images"),   (TEST_LABELS, "test/labels")]:
    count = len(list(p.iterdir())) if p.exists() else 0
    status = "✅" if count > 0 else "❌"
    print(f"  {status} {name}: {count}개 파일")

데이터셋 경로: /mnt/c/Users/user/github/yolo26_app/gym/data/dataset
클래스 수: 28
클래스 목록: ['Ab_Wheel', 'Aerobic_Stepper', 'Arm_Curl', 'Assisted_Chin_Up_Dip', 'Back_Extension', 'Barbell', 'Cable_Machine', 'Chest_Fly', 'Chest_Press', 'Clubbell', 'Dumbbell', 'Elliptical', 'Hip_Abductor', 'Kettlebell', 'Lat_Pulldown', 'Lateral_Raise', 'Leg_Curl', 'Leg_Extension', 'Leg_Press', 'Medicine_Ball', 'Plyo_Box', 'Seated_Cable_Row', 'Seated_Dip', 'Shoulder_Press', 'Smith_Machine', 'Stationary_Bike', 'T_Bar_Row', 'Treadmill']
  ✅ train/images: 37931개 파일
  ✅ train/labels: 37931개 파일
  ✅ valid/images: 3873개 파일
  ✅ valid/labels: 3873개 파일
  ✅ test/images: 3869개 파일
  ✅ test/labels: 3869개 파일


In [2]:
# 한글 폰트 설정
import platform
if platform.system() == "Windows":
    font_name = "Malgun Gothic"
elif platform.system() == "Darwin":
    font_name = "AppleGothic"
else:
    font_name = "NanumGothic"
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"한글 폰트: {font_name}")

한글 폰트: NanumGothic


---
## 유틸리티 함수

In [3]:
def parse_labels(label_dir):
    records = []
    if not label_dir.exists():
        return records
    
    for label_file in label_dir.glob("*.txt"):
        stem = label_file.stem
        
        # 파일 크기가 0바이트인 경우 (빈 라벨 처리)
        if label_file.stat().st_size == 0:
            records.append({
                "stem": stem, "class_id": -1, 
                "cx": 0.0, "cy": 0.0, "w": 0.0, "h": 0.0, "empty": True
            })
            continue
            
        with open(label_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
            
            # 내용은 없는데 빈 줄만 있는 경우
            if not lines:
                records.append({
                    "stem": stem, "class_id": -1, 
                    "cx": 0.0, "cy": 0.0, "w": 0.0, "h": 0.0, "empty": True
                })
                continue
                
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    records.append({
                        "stem": stem,
                        # 팩트 체크: '30.0' 같은 실수형 문자열 방어를 위해 float 거친 후 int 변환
                        "class_id": int(float(parts[0])), 
                        "cx": float(parts[1]),
                        "cy": float(parts[2]),
                        "w": float(parts[3]),
                        "h": float(parts[4]),
                        "empty": False,
                    })
    return records

---
# Phase 1: 심층 EDA
Phase 0 데이터 정제 완료 후, 정제된 데이터셋 기준으로 최종 EDA를 수행합니다.

## Step 1-1. 클래스별 최종 분포 확인
33개 클래스의 train/valid/test 분포가 비례적인지 확인합니다.

In [4]:
# 각 split 라벨 파싱
train_records = parse_labels(TRAIN_LABELS)
valid_records = parse_labels(VALID_LABELS)
test_records  = parse_labels(TEST_LABELS)

train_df = pd.DataFrame(train_records)
valid_df = pd.DataFrame(valid_records)
test_df  = pd.DataFrame(test_records)

# 클래스별 이미지 등장 수 (이미지 단위 — 한 이미지에 같은 클래스 여러 bbox가 있어도 1회)
def class_image_counts(df):
    valid = df[df["class_id"] >= 0]
    return valid.groupby("class_id")["stem"].nunique()

train_counts = class_image_counts(train_df)
valid_counts = class_image_counts(valid_df)
test_counts  = class_image_counts(test_df)

# 통합 테이블
dist_df = pd.DataFrame({
    "클래스": [CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"id_{i}" for i in range(NC)],
    "Train": [train_counts.get(i, 0) for i in range(NC)],
    "Valid": [valid_counts.get(i, 0) for i in range(NC)],
    "Test":  [test_counts.get(i, 0) for i in range(NC)],
})
dist_df["합계"] = dist_df["Train"] + dist_df["Valid"] + dist_df["Test"]
dist_df = dist_df.sort_values("Train", ascending=False).reset_index(drop=True)

print("=" * 70)
print("클래스별 이미지 등장 수 (Split별)")
print("=" * 70)
print(dist_df.to_string(index=False))
print(f"\nTrain 이미지 수: {train_df['stem'].nunique()}")
print(f"Valid 이미지 수: {valid_df['stem'].nunique()}")
print(f"Test 이미지 수: {test_df['stem'].nunique()}")

KeyboardInterrupt: 

In [ ]:
# 클래스별 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 1) Train 클래스 분포 바 차트
ax = axes[0]
colors = ["#e74c3c" if v < 300 else "#f39c12" if v < 500 else "#2ecc71" for v in dist_df["Train"]]
bars = ax.barh(dist_df["클래스"], dist_df["Train"], color=colors)
ax.set_xlabel("이미지 수")
ax.set_title("Train 클래스별 이미지 수", fontsize=14, fontweight="bold")
ax.invert_yaxis()
for bar, val in zip(bars, dist_df["Train"]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, str(val),
            va="center", fontsize=8)

# 범례
from matplotlib.patches import Patch
legend_items = [Patch(color="#e74c3c", label="< 300장 (희소)"),
                Patch(color="#f39c12", label="300~500장"),
                Patch(color="#2ecc71", label="> 500장")]
ax.legend(handles=legend_items, loc="lower right")

# 2) Split 비율 비교
ax2 = axes[1]
x = range(len(dist_df))
width = 0.25
ax2.bar([i - width for i in x], dist_df["Train"], width, label="Train", color="#3498db")
ax2.bar(x, dist_df["Valid"], width, label="Valid", color="#e67e22")
ax2.bar([i + width for i in x], dist_df["Test"], width, label="Test", color="#2ecc71")
ax2.set_xticks(x)
ax2.set_xticklabels(dist_df["클래스"], rotation=90, fontsize=7)
ax2.set_ylabel("이미지 수")
ax2.set_title("Train / Valid / Test 분포 비교", fontsize=14, fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.show()

# 비율 균형 체크
print("\n── Split 비율 체크 ──")
total = dist_df["합계"].sum()
for col in ["Train", "Valid", "Test"]:
    pct = dist_df[col].sum() / total * 100
    print(f"  {col}: {dist_df[col].sum()}장 ({pct:.1f}%)")

## Step 1-1-2. 오버샘플링 / 증강 반영 확인
복사(_os) 및 Albumentations 증강(_alb) 이미지가 정상적으로 반영되었는지 확인합니다.

In [ ]:
train_stems = [f.stem for f in TRAIN_LABELS.glob("*.txt")]

original = [s for s in train_stems if "_os" not in s and "_alb" not in s]
oversampled = [s for s in train_stems if "_os" in s]
augmented = [s for s in train_stems if "_alb" in s]

print("── 오버샘플링 / 증강 현황 ──")
print(f"  원본 이미지:       {len(original):>6,}장")
print(f"  오버샘플링 복사:   {len(oversampled):>6,}장 (_os 접미사)")
print(f"  Albumentations:    {len(augmented):>6,}장 (_alb 접미사)")
print(f"  ────────────────────────────")
print(f"  합계:              {len(train_stems):>6,}장")

# 증강 이미지가 어떤 클래스에 속하는지 확인
aug_class_counter = Counter()
for stem in oversampled + augmented:
    label_path = TRAIN_LABELS / f"{stem}.txt"
    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    # 팩트 체크: '30.0' 형태의 문자열 에러를 막기 위한 이중 캐스팅 적용
                    cid = int(float(parts[0]))
                    if 0 <= cid < NC:
                        aug_class_counter[CLASS_NAMES[cid]] += 1
                    break  # 이미지당 대표 클래스 1개만

print("\n── 증강 이미지의 클래스 분포 ──")
for cls, cnt in aug_class_counter.most_common():
    print(f"  {cls}: {cnt}장")

## Step 1-1-3. BBox 크기 분포 분석
소형 객체(bbox 면적 < 32² 픽셀) 비율을 파악하여 YOLO26의 STAL 손실 함수 효과를 예측합니다.

In [ ]:
# 이미지 크기 640x640 기준 bbox 픽셀 크기 계산
IMG_SIZE = 640
train_valid = train_df[train_df["class_id"] >= 0].copy()
train_valid["w_px"] = train_valid["w"] * IMG_SIZE
train_valid["h_px"] = train_valid["h"] * IMG_SIZE
train_valid["area_px"] = train_valid["w_px"] * train_valid["h_px"]

# COCO 기준 크기 분류
train_valid["size_cat"] = pd.cut(
    train_valid["area_px"],
    bins=[0, 32**2, 96**2, float("inf")],
    labels=["Small (<32²)", "Medium (32²~96²)", "Large (>96²)"]
)

size_dist = train_valid["size_cat"].value_counts()
print("── BBox 크기 분포 (COCO 기준) ──")
for cat, cnt in size_dist.items():
    pct = cnt / len(train_valid) * 100
    print(f"  {cat}: {cnt}개 ({pct:.1f}%)")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) 크기 파이 차트
axes[0].pie(size_dist.values, labels=size_dist.index, autopct="%1.1f%%",
            colors=["#e74c3c", "#f39c12", "#2ecc71"], startangle=90)
axes[0].set_title("BBox 크기 분포", fontsize=13, fontweight="bold")

# 2) BBox 너비 vs 높이 산점도
sample = train_valid.sample(min(5000, len(train_valid)), random_state=42)
axes[1].scatter(sample["w_px"], sample["h_px"], alpha=0.15, s=3, c="#3498db")
axes[1].set_xlabel("Width (px)")
axes[1].set_ylabel("Height (px)")
axes[1].set_title("BBox 너비 vs 높이", fontsize=13, fontweight="bold")
axes[1].axhline(32, color="red", linestyle="--", alpha=0.5, label="32px")
axes[1].axvline(32, color="red", linestyle="--", alpha=0.5)
axes[1].legend()

# 3) 클래스별 평균 BBox 면적
cls_area = train_valid.groupby("class_id")["area_px"].mean().sort_values()
cls_names_sorted = [CLASS_NAMES[i] if i < NC else f"id_{i}" for i in cls_area.index]
axes[2].barh(cls_names_sorted, cls_area.values, color="#9b59b6", alpha=0.8)
axes[2].set_xlabel("평균 BBox 면적 (px²)")
axes[2].set_title("클래스별 평균 BBox 면적", fontsize=13, fontweight="bold")
axes[2].axvline(32**2, color="red", linestyle="--", alpha=0.5, label="32² (Small 경계)")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\n📌 Small 객체 비율: {(train_valid['size_cat'] == 'Small (<32²)').mean()*100:.1f}%")
print("   → YOLO26의 STAL(Small-Target-Aware Label Assignment)이 이 비율만큼 효과를 발휘합니다.")

## Step 1-1-4. 데이터셋 간 중복 검사
7개 소스 데이터셋 간 이미지 해시를 비교하여 train-valid-test 누수(leakage)가 없는지 확인합니다.

> ⚠️ 이 작업은 전체 이미지 파일을 읽어 해시를 계산하므로 수 분이 소요될 수 있습니다.

In [ ]:
from hashlib import md5

def compute_hashes(images_dir: Path, sample_size: int = None) -> dict:
    """이미지 파일의 전체 해시를 계산합니다."""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = [f for f in sorted(images_dir.iterdir()) if f.suffix.lower() in exts]
    if sample_size and sample_size < len(files):
        files = random.sample(files, sample_size)
    
    hash_map = {}
    for f in files:
        with open(f, "rb") as fp:
            h = md5(fp.read()).hexdigest()
        hash_map[f.name] = h
    return hash_map

print("해시 계산 중... (전체 이미지)")
train_hashes = compute_hashes(TRAIN_IMAGES)
valid_hashes = compute_hashes(VALID_IMAGES)
test_hashes  = compute_hashes(TEST_IMAGES)

# 역매핑: hash → [(split, filename), ...]
hash_to_files = defaultdict(list)
for name, h in train_hashes.items():
    hash_to_files[h].append(("train", name))
for name, h in valid_hashes.items():
    hash_to_files[h].append(("valid", name))
for name, h in test_hashes.items():
    hash_to_files[h].append(("test", name))

# 크로스 스플릿 중복 탐지
cross_leaks = []
for h, files in hash_to_files.items():
    splits = set(s for s, _ in files)
    if len(splits) > 1:
        cross_leaks.append(files)

print(f"\n── 중복 검사 결과 ──")
print(f"  Train 이미지: {len(train_hashes)}장")
print(f"  Valid 이미지: {len(valid_hashes)}장")
print(f"  Test  이미지: {len(test_hashes)}장")

if cross_leaks:
    print(f"\n  ⚠️ 크로스 스플릿 중복: {len(cross_leaks)}건")
    print(f"  (동일한 이미지가 train/valid/test에 동시 존재)")
    for leak in cross_leaks[:5]:
        print(f"    {leak}")
    if len(cross_leaks) > 5:
        print(f"    ... 외 {len(cross_leaks) - 5}건")
    print(f"\n  💡 640×640 리사이징 시 파일 헤더가 유사해져서 발생하는 오탐일 수 있습니다.")
    print(f"     파일명이 서로 다른 소스(d1, d2 등)에서 왔다면 실제 누수가 아닐 가능성이 높습니다.")
else:
    print(f"\n  ✅ 크로스 스플릿 중복 없음!")

## Phase 1 요약
위 EDA 결과를 확인하고 이상이 없으면 Phase 2 학습으로 넘어갑니다.

### EDA에서 발견된 문제 및 조치 (fix_eda_issues.py로 처리 완료)
| 문제 | 조치 | 상태 |
|---|---|---|
| 세그멘테이션 라벨 5개 혼입 | bbox(5컬럼)로 자름 + .cache 삭제 | ✅ 완료 |
| Foam_Roller valid/test 0장 | train 원본에서 valid 7장, test 7장 이동 | ✅ 완료 |
| Dumbbell_Rack valid 9장 (극소) | train 원본에서 valid로 5장 추가 이동 | ✅ 완료 |

### 수정 불필요 확인 항목
- ✅ 전체 640×640 고정 → imgsz=640 최적
- ✅ Large BBox 90.5% → STAL 의존도 낮음, 기본 설정 적절
- ✅ Split 비율 83/8/8 → 정상
- ✅ 증강 설정 (Mosaic 1.0, MixUp 0.0) 유지 적절

---
# Phase 2: YOLO26 모델 학습 — 다중 실험

> **실험 설계 (4개)**

| 실험 | 모델 | 초기 가중치 | Epochs | Patience | Batch | 목적 |
|------|------|-----------|--------|----------|-------|------|
| 베이스라인 | yolo26s | yolo26s.pt (COCO) | 15 (조기종료) | 7 | 48 | 기존 결과 유지 |
| 실험 A | yolo26s | yolo26s.pt (COCO) | 100 | 20 | 48 | epoch 증가 효과 |
| 실험 B | yolo26m | yolo26m.pt (COCO) | 100 | 20 | 24 | 모델 스케일업 효과 |
| 실험 C | yolo26s | small.pt (커스텀) | 100 | 20 | 48 | 전이학습 비교 |

> ⚠️ 실험 B는 yolo26m(20.4M params)이므로 batch를 24로 줄임 (VRAM 16GB 기준)  
> ⚠️ 실험 C의 `small.pt`는 다른 참가자가 1개 데이터셋(12클래스, 512px)으로 학습한 가중치

## Step 2-0. 실험 설정 및 공통 파라미터

> 실험 B는 20 epoch마다 중간 저장

In [ ]:
from ultralytics import YOLO

# ──────────────────────────────────────────────
# ⚙️ 실험 설정 — 모든 실험의 공통/개별 파라미터
# ──────────────────────────────────────────────

PROJECT_DIR = str(DATASET_ROOT / "runs" / "detect")

# small.pt 경로 (실험 C용 — 다른 참가자의 커스텀 가중치)
# 이 파일이 없으면 실험 C는 자동 스킵됩니다
CUSTOM_SMALL_PT = DATASET_ROOT / "pretrained" / "small.pt"

EXPERIMENTS = {
    "baseline": {
        "model": "yolo26s.pt",
        "run_name": "gym_yolo26s_baseline",
        "epochs": 30,
        "patience": 7,
        "batch": 48,
        "time": 1.8,       # 기존 베이스라인과 동일 조건
        "description": "베이스라인 (yolo26s, 15 epochs 조기종료)",
    },
    "exp_A": {
        "model": "yolo26s.pt",
        "run_name": "gym_yolo26s_100ep",
        "epochs": 100,
        "patience": 20,
        "batch": 48,
        "time": None,       # 시간 제한 없음
        "description": "실험 A: yolo26s + 100 epochs",
    },
    "exp_B": {
        "model": "yolo26m.pt",
        "run_name": "gym_yolo26m_100ep",
        "epochs": 100,
        "patience": 20,
        "batch": 24,        
        "time": None,
        "save_period": 20,  # 👈 추가: 20 에포크마다 중간 가중치 저장
        "description": "실험 B: yolo26m + 100 epochs (스케일업)",
    },
    "exp_C": {
        "model": str(CUSTOM_SMALL_PT) if CUSTOM_SMALL_PT.exists() else None,
        "run_name": "gym_yolo26s_transfer",
        "epochs": 100,
        "patience": 20,
        "batch": 48,
        "time": None,
        "description": "실험 C: yolo26s + small.pt pretrained (전이학습)",
    },
}

# 공통 증강 설정
COMMON_AUG = dict(
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
)

# 공통 학습 설정
COMMON_TRAIN = dict(
    data=str(DATA_YAML),
    imgsz=640,
    optimizer="auto",
    workers=12,
    cache="ram",
    device=0,
    amp=True,
    plots=True,
    verbose=True,
)

# 실험 상태 확인
print("=" * 60)
print("실험 설정 요약")
print("=" * 60)
for exp_id, cfg in EXPERIMENTS.items():
    best_pt = Path(PROJECT_DIR) / cfg["run_name"] / "weights" / "best.pt"
    status = "✅ 학습 완료" if best_pt.exists() else "⏳ 대기"
    model_status = "❌ 가중치 없음" if cfg["model"] is None else cfg["model"]
    print(f"  {exp_id:12s} | {cfg['description']}")
    print(f"{'':14s} | 모델: {model_status} | batch: {cfg['batch']} | epochs: {cfg['epochs']} | {status}")
    print()

## Step 2-1. 데이터 무결성 확인

In [ ]:
# ──────────────────────────────────────────────
# 🔧 데이터 무결성 및 캐시 초기화 확인
# ──────────────────────────────────────────────

# 1. 세그멘테이션 라벨(좌표가 5개를 초과하는 라벨) 잔존 여부 확인
seg_count = 0
for split in ["train", "valid", "test"]:
    labels_dir = DATASET_ROOT / split / "labels"
    if not labels_dir.exists():
        continue
    for label_path in labels_dir.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                if len(line.strip().split()) > 5:
                    seg_count += 1
                    break

# 2. YOLO 캐시 파일(.cache) 잔존 확인
cache_files = list(DATASET_ROOT.rglob("*.cache"))

print("── 데이터 무결성 확인 ──")
print(f"  세그멘테이션 라벨 잔존: {seg_count}개 {'✅ 정리 완료' if seg_count == 0 else '❌ 불량 라벨 삭제 필요'}")
print(f"  .cache 파일: {len(cache_files)}개 {'✅ 삭제 완료' if len(cache_files) == 0 else '⚠️ 캐시 잔존 (삭제 권장)'}")

# 만약 캐시가 남아있다면 파이썬 코드로 즉시 삭제 처리
if len(cache_files) > 0:
    print("\n⚠️ 잔존하는 .cache 파일을 강제 삭제합니다...")
    for c_file in cache_files:
        try:
            c_file.unlink()
            print(f" - 삭제됨: {c_file.name}")
        except Exception as e:
            print(f" - 삭제 실패: {c_file.name} ({e})")

## Step 2-2. 실험 실행

> 각 실험은 `best.pt`가 이미 존재하면 자동으로 건너뜁니다.  
> 특정 실험만 재실행하려면 해당 `runs/detect/{run_name}` 폴더를 삭제하세요.

In [ ]:
# ──────────────────────────────────────────────
# 🚀 전체 실험 순차 실행 (save_period 반영)
# ──────────────────────────────────────────────

for exp_id, cfg in EXPERIMENTS.items():
    print(f"\n{'='*60}")
    print(f"🧪 {cfg['description']}")
    print(f"{'='*60}")
    
    if cfg["model"] is None:
        print(f"  ⚠️ 초기 가중치 파일이 없습니다. 건너뜁니다.")
        continue
    
    best_pt = Path(PROJECT_DIR) / cfg["run_name"] / "weights" / "best.pt"
    
    if best_pt.exists():
        print(f"  ✅ 이미 학습 완료: {best_pt}")
        continue
    
    model = YOLO(cfg["model"])
    
    train_args = {
        **COMMON_TRAIN,
        **COMMON_AUG,
        "epochs": cfg["epochs"],
        "batch": cfg["batch"],
        "patience": cfg["patience"],
        "project": PROJECT_DIR,
        "name": cfg["run_name"],
        "exist_ok": True,
    }
    
    if cfg.get("time"):
        train_args["time"] = cfg["time"]
        
    # 👈 추가된 부분: 설정에 save_period가 있으면 파라미터로 넘겨줌
    if "save_period" in cfg:
        train_args["save_period"] = cfg["save_period"]
        print(f"  💾 자동 저장: 매 {cfg['save_period']} 에포크마다 백업")
    
    results = model.train(**train_args)
    
    print(f"\n  ✅ {exp_id} 학습 완료!")

print("\n🏁 전체 실험 완료!")

## Step 2-3. 모델 로드 (평가용)

In [ ]:
# ──────────────────────────────────────────────
# 📂 학습된 모델 로드 — 모든 실험의 best.pt
# ──────────────────────────────────────────────
from ultralytics import YOLO

models = {}
for exp_id, cfg in EXPERIMENTS.items():
    best_pt = Path(PROJECT_DIR) / cfg["run_name"] / "weights" / "best.pt"
    if best_pt.exists():
        models[exp_id] = YOLO(str(best_pt))
        print(f"  ✅ {exp_id}: {best_pt.name} 로드 완료")
        models[exp_id].info()
    else:
        print(f"  ❌ {exp_id}: best.pt 없음 (학습 미완료)")
    print()

print(f"\n로드된 모델: {list(models.keys())}")

---
# Phase 3: 성능 검증 및 실험 비교

## Step 3-1. 전체 실험 성능 평가

In [ ]:
# ──────────────────────────────────────────────
# 📊 모든 실험의 Test 셋 평가
# ──────────────────────────────────────────────

all_metrics = {}

for exp_id, model in models.items():
    print(f"\n{'='*60}")
    print(f"📊 {EXPERIMENTS[exp_id]['description']} — Test 셋 평가")
    print(f"{'='*60}")
    
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        batch=64,
        device=0,
        plots=True,
        verbose=True,
    )
    
    all_metrics[exp_id] = metrics
    
    print(f"\n  mAP@50:    {metrics.box.map50:.4f}")
    print(f"  mAP@50:95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall:    {metrics.box.mr:.4f}")

print(f"\n✅ 평가 완료: {list(all_metrics.keys())}")

## Step 3-2. 실험 간 전체 성능 비교

In [ ]:
# ──────────────────────────────────────────────
# 📊 실험 간 전체 성능 비교 테이블 + 바 차트
# ──────────────────────────────────────────────

compare_rows = []
for exp_id, metrics in all_metrics.items():
    compare_rows.append({
        "실험": exp_id,
        "설명": EXPERIMENTS[exp_id]["description"],
        "mAP@50": round(metrics.box.map50, 4),
        "mAP@50:95": round(metrics.box.map, 4),
        "Precision": round(metrics.box.mp, 4),
        "Recall": round(metrics.box.mr, 4),
    })

compare_df = pd.DataFrame(compare_rows)
print("=" * 80)
print("실험 간 전체 성능 비교")
print("=" * 80)
print(compare_df.to_string(index=False))

# 바 차트
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metric_cols = ["mAP@50", "mAP@50:95", "Precision", "Recall"]
colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]

for ax, col, color in zip(axes, metric_cols, colors):
    bars = ax.bar(compare_df["실험"], compare_df[col], color=color, alpha=0.8)
    ax.set_title(col, fontsize=13, fontweight="bold")
    ax.set_ylim(0.7, 1.0)
    ax.set_ylabel(col)
    for bar, val in zip(bars, compare_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle("실험 간 전체 성능 비교", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 3-3. 하위 클래스 성능 비교 (핵심)

In [ ]:
# ──────────────────────────────────────────────
# 📊 하위 5개 클래스 실험 간 비교 (Dumbbell, Elliptical 등)
# ──────────────────────────────────────────────

FOCUS_CLASSES = ["Dumbbell", "Elliptical", "Hip_Abductor", "Medicine_Ball", "Kettlebell"]

focus_rows = []
for exp_id, metrics in all_metrics.items():
    for cls_name in FOCUS_CLASSES:
        if cls_name in CLASS_NAMES:
            idx = CLASS_NAMES.index(cls_name)
            focus_rows.append({
                "실험": exp_id,
                "클래스": cls_name,
                "mAP@50": metrics.box.ap50[idx],
                "mAP@50:95": metrics.box.maps[idx],
                "Recall": metrics.box.r[idx] if hasattr(metrics.box, 'r') else 0,
            })

focus_df = pd.DataFrame(focus_rows)
print("=" * 80)
print("하위 클래스 실험 간 mAP@50 비교")
print("=" * 80)

pivot = focus_df.pivot_table(index="클래스", columns="실험", values="mAP@50")
print(pivot.to_string())

# 그룹 바 차트
fig, ax = plt.subplots(figsize=(14, 7))
pivot.plot(kind="bar", ax=ax, width=0.7, alpha=0.85)
ax.set_title("하위 클래스 — 실험 간 mAP@50 비교", fontsize=14, fontweight="bold")
ax.set_ylabel("mAP@50")
ax.set_ylim(0.5, 1.0)
ax.axhline(0.90, color="red", linestyle="--", alpha=0.5, label="목표 (0.90)")
ax.legend(title="실험", loc="lower right")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=7, padding=2)

plt.tight_layout()
plt.show()

## Step 3-4. 클래스별 AP@50 상세 (최고 성능 모델)

In [ ]:
# ──────────────────────────────────────────────
# 📊 최고 성능 모델의 클래스별 AP@50
# ──────────────────────────────────────────────

# mAP@50 기준 최고 실험 선택
best_exp = max(all_metrics.keys(), key=lambda k: all_metrics[k].box.map50)
best_metrics = all_metrics[best_exp]
print(f"🏆 최고 성능 모델: {best_exp} ({EXPERIMENTS[best_exp]['description']})")
print(f"   mAP@50: {best_metrics.box.map50:.4f}")

class_ap50 = best_metrics.box.ap50
ap_df = pd.DataFrame({
    "클래스": CLASS_NAMES,
    "AP@50": class_ap50,
}).sort_values("AP@50", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 10))
colors = ["#e74c3c" if v < 0.8 else "#f39c12" if v < 0.9 else "#2ecc71" for v in ap_df["AP@50"]]
bars = ax.barh(ap_df["클래스"], ap_df["AP@50"], color=colors)
ax.set_xlabel("AP@50")
ax.set_title(f"클래스별 AP@50 — {best_exp} (Test Set)", fontsize=14, fontweight="bold")
ax.axvline(0.9, color="red", linestyle="--", alpha=0.5, label="목표 (0.90)")

for bar, val in zip(bars, ap_df["AP@50"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8)

legend_items = [Patch(color="#e74c3c", label="AP < 0.80"),
                Patch(color="#f39c12", label="0.80 ≤ AP < 0.90"),
                Patch(color="#2ecc71", label="AP ≥ 0.90")]
ax.legend(handles=legend_items, loc="lower right")
plt.tight_layout()
plt.show()

## Step 3-4.5. 실험별 학습 곡선 시각화
각 실험의 `results.csv`에서 loss, precision, recall, mAP 추이를 그래프로 비교합니다.

In [ ]:
# ──────────────────────────────────────────────
# 📈 실험별 학습 곡선 (results.csv) 시각화
# ──────────────────────────────────────────────
import pandas as pd

# 각 실험의 results.csv 로드
results_dfs = {}
for exp_id, cfg in EXPERIMENTS.items():
    csv_path = Path(PROJECT_DIR) / cfg["run_name"] / "results.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()  # 컬럼명 공백 제거
        results_dfs[exp_id] = df
        print(f"  ✅ {exp_id}: {len(df)} epochs ({csv_path.name})")
    else:
        print(f"  ❌ {exp_id}: results.csv 없음")

print(f"\n로드된 실험: {list(results_dfs.keys())}") 

In [ ]:
# ──────────────────────────────────────────────
# 📈 개별 실험 학습 곡선 (10개 지표 — 첨부 이미지와 동일 형태)
# ──────────────────────────────────────────────

# 시각화할 지표 (results.csv 컬럼명)
plot_metrics = [
    ("train/box_loss",    "Train Box Loss"),
    ("train/cls_loss",    "Train Cls Loss"),
    ("train/dfl_loss",    "Train DFL Loss"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)",    "Recall"),
    ("val/box_loss",      "Val Box Loss"),
    ("val/cls_loss",      "Val Cls Loss"),
    ("val/dfl_loss",      "Val DFL Loss"),
    ("metrics/mAP50(B)",     "mAP@50"),
    ("metrics/mAP50-95(B)",  "mAP@50-95"),
]

for exp_id, df in results_dfs.items():
    fig, axes = plt.subplots(2, 5, figsize=(24, 8))
    axes = axes.flatten()
    
    for ax, (col, title) in zip(axes, plot_metrics):
        if col in df.columns:
            epochs = range(1, len(df) + 1)
            values = df[col].values
            
            ax.plot(epochs, values, 'o-', markersize=2, linewidth=1, color='#1f77b4', label='results')
            
            # 스무딩 (EMA)
            if len(values) > 3:
                alpha = 0.3
                smoothed = [values[0]]
                for v in values[1:]:
                    smoothed.append(alpha * v + (1 - alpha) * smoothed[-1])
                ax.plot(epochs, smoothed, '--', linewidth=1.5, color='#ff7f0e', alpha=0.8, label='smooth')
            
            ax.set_title(title, fontsize=11, fontweight='bold')
            ax.set_xlabel('Epoch', fontsize=8)
            ax.grid(True, alpha=0.3)
            
            if ax == axes[0]:
                ax.legend(fontsize=8)
        else:
            ax.set_title(f"{title}\n(컬럼 없음)", fontsize=10, color='red')
            ax.axis('off')
    
    plt.suptitle(f"학습 곡선 — {exp_id} ({EXPERIMENTS[exp_id]['description']})",
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ──────────────────────────────────────────────
# 📈 실험 간 핵심 지표 비교 (같은 그래프에 오버레이)
# ──────────────────────────────────────────────

compare_metrics = [
    ("train/box_loss",       "Train Box Loss"),
    ("val/box_loss",         "Val Box Loss"),
    ("metrics/mAP50(B)",     "mAP@50"),
    ("metrics/mAP50-95(B)",  "mAP@50-95"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)",    "Recall"),
]

colors = {'baseline': '#7f7f7f', 'exp_A': '#1f77b4', 'exp_B': '#ff7f0e', 'exp_C': '#2ca02c'}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for ax, (col, title) in zip(axes, compare_metrics):
    for exp_id, df in results_dfs.items():
        if col in df.columns:
            epochs = range(1, len(df) + 1)
            ax.plot(epochs, df[col].values, '-', linewidth=1.5,
                    color=colors.get(exp_id, '#333'),
                    label=exp_id, alpha=0.85)
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("실험 간 학습 곡선 비교", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3-5. 유사 기구 혼동 분석
형태가 유사한 클래스 쌍의 교차 오분류율을 확인합니다.

- Chest_Press ↔ Shoulder_Press
- Leg_Curl ↔ Leg_Extension
- Elliptical ↔ Stationary_Bike
- Aerobic_Stepper ↔ Plyo_Box
- Kettlebell ↔ Dumbbell
- **Dumbbell ↔ Barbell** (신규 추가)

In [ ]:
# ──────────────────────────────────────────────
# 🔄 Confusion Matrix 혼동 분석 — 최고 성능 모델 기준
# ──────────────────────────────────────────────

# 최고 성능 모델 선택
best_exp = max(all_metrics.keys(), key=lambda k: all_metrics[k].box.map50)
best_metrics = all_metrics[best_exp]
best_model = models[best_exp]
print(f"🏆 최고 성능 모델: {best_exp} ({EXPERIMENTS[best_exp]['description']})")
print(f"   mAP@50: {best_metrics.box.map50:.4f} | mAP@50-95: {best_metrics.box.map:.4f}")

confusion_matrix = best_metrics.confusion_matrix.matrix

confuse_pairs = [
    ("Chest_Press", "Shoulder_Press"),
    ("Leg_Curl", "Leg_Extension"),
    ("Elliptical", "Stationary_Bike"),
    ("Aerobic_Stepper", "Plyo_Box"),
    ("Kettlebell", "Dumbbell"),
    ("Dumbbell", "Barbell"),
]

print(f"\n{'='*70}")
print(f"유사 기구 혼동 분석 — {best_exp}")
print(f"{'='*70}")

for cls_a, cls_b in confuse_pairs:
    idx_a = CLASS_NAMES.index(cls_a) if cls_a in CLASS_NAMES else -1
    idx_b = CLASS_NAMES.index(cls_b) if cls_b in CLASS_NAMES else -1
    
    if idx_a < 0 or idx_b < 0:
        print(f"  ⚠️ {cls_a} ↔ {cls_b}: 인덱스를 찾을 수 없음")
        continue
    
    a_as_b = confusion_matrix[idx_a][idx_b]
    b_as_a = confusion_matrix[idx_b][idx_a]
    total_a = confusion_matrix[idx_a].sum()
    total_b = confusion_matrix[idx_b].sum()
    
    rate_a_as_b = (a_as_b / total_a * 100) if total_a > 0 else 0
    rate_b_as_a = (b_as_a / total_b * 100) if total_b > 0 else 0
    
    warning = " ⚠️" if rate_a_as_b > 5 or rate_b_as_a > 5 else ""
    
    print(f"\n  {cls_a} ↔ {cls_b}{warning}")
    print(f"    {cls_a} → {cls_b}로 오분류: {a_as_b:.0f}건 ({rate_a_as_b:.1f}%)")
    print(f"    {cls_b} → {cls_a}로 오분류: {b_as_a:.0f}건 ({rate_b_as_a:.1f}%)")

In [ ]:
# ──────────────────────────────────────────────
# 📊 Confusion Matrix 히트맵 — 최고 성능 모델 기준
# ──────────────────────────────────────────────

focus_classes = sorted(set(cls for pair in confuse_pairs for cls in pair))
focus_indices = [CLASS_NAMES.index(c) for c in focus_classes if c in CLASS_NAMES]
focus_names = [CLASS_NAMES[i] for i in focus_indices]

sub_matrix = confusion_matrix[np.ix_(focus_indices, focus_indices)]
row_sums = sub_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
norm_matrix = sub_matrix / row_sums

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(norm_matrix, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(focus_names)))
ax.set_yticks(range(len(focus_names)))
ax.set_xticklabels(focus_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(focus_names, fontsize=9)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground Truth")
ax.set_title(f"유사 기구 혼동 매트릭스 — {best_exp}", fontsize=14, fontweight="bold")

for i in range(len(focus_names)):
    for j in range(len(focus_names)):
        val = norm_matrix[i][j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=color)

plt.colorbar(im, label="비율")
plt.tight_layout()
plt.show()

## Step 3-6. 실험 간 Dumbbell ↔ Barbell 혼동률 비교

In [ ]:
# ──────────────────────────────────────────────
# 📊 Dumbbell ↔ Barbell 혼동률 — 실험 간 비교
# ──────────────────────────────────────────────

db_idx = CLASS_NAMES.index("Dumbbell") if "Dumbbell" in CLASS_NAMES else -1
bb_idx = CLASS_NAMES.index("Barbell") if "Barbell" in CLASS_NAMES else -1

if db_idx >= 0 and bb_idx >= 0:
    confuse_rows = []
    for exp_id, metrics in all_metrics.items():
        cm = metrics.confusion_matrix.matrix
        
        db_total = cm[db_idx].sum()
        bb_total = cm[bb_idx].sum()
        
        db_as_bb = cm[db_idx][bb_idx]
        bb_as_db = cm[bb_idx][db_idx]
        
        confuse_rows.append({
            "실험": exp_id,
            "Dumbbell→Barbell": f"{db_as_bb:.0f}건 ({db_as_bb/db_total*100:.1f}%)" if db_total > 0 else "-",
            "Barbell→Dumbbell": f"{bb_as_db:.0f}건 ({bb_as_db/bb_total*100:.1f}%)" if bb_total > 0 else "-",
            "Dumbbell mAP50": f"{metrics.box.ap50[db_idx]:.3f}",
            "Barbell mAP50": f"{metrics.box.ap50[bb_idx]:.3f}",
        })
    
    confuse_df = pd.DataFrame(confuse_rows)
    print("=" * 80)
    print("Dumbbell ↔ Barbell 혼동률 — 실험 간 비교")
    print("=" * 80)
    print(confuse_df.to_string(index=False))
else:
    print("❌ Dumbbell 또는 Barbell 클래스를 찾을 수 없습니다.")

## Step 3-7. Test 이미지 추론 시각화
최고 성능 모델로 Test 이미지를 추론합니다. 배경 이미지(삭제된 클래스)는 제외됩니다.

In [ ]:
# ──────────────────────────────────────────────
# 🖼️ Test 이미지 추론 시각화 — 최고 성능 모델 기준
# ──────────────────────────────────────────────
import cv2
import random

# 배경 이미지(빈 라벨) 제외
test_imgs_all = list(TEST_IMAGES.glob("*.jpg")) + list(TEST_IMAGES.glob("*.png"))
test_imgs = []
for img_path in test_imgs_all:
    lbl_path = TEST_LABELS / (img_path.stem + ".txt")
    if lbl_path.exists() and lbl_path.stat().st_size > 0:
        with open(lbl_path, "r") as f:
            if f.read().strip():
                test_imgs.append(img_path)

print(f"전체 Test: {len(test_imgs_all)}장 → 라벨 있는 이미지: {len(test_imgs)}장")
print(f"🏆 추론 모델: {best_exp} ({EXPERIMENTS[best_exp]['description']})")

samples = random.sample(test_imgs, min(9, len(test_imgs)))

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

for idx, img_path in enumerate(samples):
    results = best_model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)
    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    axes[idx].imshow(result_img)
    axes[idx].set_title(img_path.name[:30], fontsize=9)
    axes[idx].axis("off")

for idx in range(len(samples), 9):
    axes[idx].axis("off")

plt.suptitle(f"Test 추론 결과 — {best_exp} (conf ≥ 0.25)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Phase 1~3 완료 요약

### Phase 1 (EDA)
- ✅ 28개 클래스 train/valid/test 분포 확인
- ✅ 오버샘플링 + 증강 반영 확인
- ✅ BBox 크기 분포 분석
- ✅ 데이터 누수 검사
- ✅ Workout Pose 이미지 삭제 (`remove_workout_pose.py`)

### Phase 2 (학습)
- ✅ 베이스라인: yolo26s / 15 epochs / patience 7
- ✅ 실험 A: yolo26s / 100 epochs / patience 20
- ✅ 실험 B: yolo26m / 100 epochs / patience 20
- ⬜ 실험 C: yolo26s + small.pt / 100 epochs (선택)

### Phase 3 (검증)
- ✅ 전체 실험 성능 비교 (mAP@50, mAP@50:95, Precision, Recall)
- ✅ 하위 클래스 성능 비교 (Dumbbell, Elliptical, Hip_Abductor, Medicine_Ball, Kettlebell)
- ✅ 유사 기구 혼동 분석 (Dumbbell ↔ Barbell 추가)
- ✅ Test 이미지 추론 시각화